# Calc Offset v1
Load car parquet data with selected columns

In [ ]:
import pandas as pd
import json
import os
from glm_model_test import load_glm_model, predict_glm
from default_value_handler import check_missing_values, apply_all_defaults

In [ ]:
# Load config
with open('config.json', 'r') as f:
    config = json.load(f)

parquet_path = config['car_parquet_path']
print(f"Parquet path: {parquet_path}")
short_path = '/'.join(parquet_path.split('/')[-3:])
print(f"Parquet path: {short_path}")

vin_bsst_path = config['vin_bsst_path']
print(f"vin_bsst_path path: {vin_bsst_path}")
short_vin_bsst_path = '/'.join(vin_bsst_path.split('/')[-3:])
print(f"Parquet path: {short_vin_bsst_path}")

# Load BSST_GLM_path from config
bsst_glm_path = config['BSST_GLM_path']
print(f"BSST GLM path: {bsst_glm_path}")



In [ ]:
# Define columns to load
columns = [
    'cef_est_curr_mi_grp_imps', # 'odometer',
    'zip_pop_dens',#'geo_pop_density_ntile',
    'dml_year_imps', #'CALC_VEH_AGE',
    'vc_msrp_impa',
    'st_raw',
    'insstate',
    'dml_make_raw',
    'vin',
    'vin_date', 
    'zip'
]

# Load parquet file with selected columns
df = pd.read_parquet(parquet_path, columns=columns)
print(f"Data loaded successfully!")

In [ ]:
# Display basic info
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:")
print(df.dtypes)

In [ ]:
# Preview data
df.head()

In [ ]:
# Count records where cef_est_curr_mi_grp_imps = 0
zero_count = (df['cef_est_curr_mi_grp_imps'] == 0).sum()
total_count = len(df)
print(f"Records with cef_est_curr_mi_grp_imps = 0: {zero_count:,}")
print(f"Total records: {total_count:,}")
print(f"Percentage: {zero_count/total_count*100:.2f}%")



In [ ]:
odo_median = df['cef_est_curr_mi_grp_imps'].median()
print(odo_median) 

In [ ]:
# Load VIN_BSST parquet file
df_bsst = pd.read_parquet(vin_bsst_path, columns=['VIN', 'BODY_STYLE_SEGMENT_BODY_TYPE'])
print(f"VIN_BSST loaded: {df_bsst.shape}")
df_bsst.head()

In [ ]:
# Merge BODY_STYLE_SEGMENT_BODY_TYPE to df using VIN
df = df.merge(df_bsst, left_on='vin', right_on='VIN', how='left')
print(f"After merge shape: {df.shape}")

In [ ]:
# Create BSST_formatted column
# Transform: "Basic Economy (Car)" -> "Basic_Economy_Car_GLM"
df['BSST_formatted'] = (df['BODY_STYLE_SEGMENT_BODY_TYPE']
    .str.replace(' ', '_')
    .str.replace('_(', '_', regex=False)
    .str.replace(')', '', regex=False) + '_GLM')

# Preview the formatting
print("Sample BSST_formatted values:")
print(df[['BODY_STYLE_SEGMENT_BODY_TYPE', 'BSST_formatted']].drop_duplicates().head(10))

In [ ]:
# Get unique BSST_formatted values
unique_bsst = df['BSST_formatted'].dropna().unique()
print(f"Found {len(unique_bsst)} unique BSST_formatted values:")
for bsst in sorted(unique_bsst):
    print(f"  - {bsst}")

## Data Quality Check - Missing Values
Check for zeros, nulls, empty strings for key features before applying defaults

In [ ]:
# Check missing values BEFORE applying defaults
print("Model Year from config:", config['model_year'])
missing_summary_before = check_missing_values(df, config['model_year'])

## Apply Default Values
Fill missing values using lookup tables:
- **Odometer**: Lookup by vehicle age (ages > 25 use age 25 default)
- **State**: Lookup by BSST (Body Style Segment Body Type)
- **Population Density Percentile**: Use 50 (median) as default

In [ ]:
# Apply all default values
df, fill_counts = apply_all_defaults(df, config)

In [ ]:
# Check missing values AFTER applying defaults
missing_summary_after = check_missing_values(df, config['model_year'])

## Process Full Data - GLM Predictions
Run GLM predictions for ALL BSST segments and combine results

In [ ]:
def prepare_and_predict(df_segment, glm_model, model_year):
    """
    Prepare data for GLM model and run predictions for a single BSST segment.
    
    Parameters:
    -----------
    df_segment : pd.DataFrame
        Data filtered for a specific BSST segment
    glm_model : dict
        Loaded GLM model configuration
    model_year : int
        Model year for calculating vehicle age
        
    Returns:
    --------
    pd.Series
        Predictions (Dep_factor) for all records in the segment
    """
    df_pred = df_segment.copy()
    
    # Map columns to GLM model expected names
    df_pred['ODOMETER'] = df_pred['cef_est_curr_mi_grp_imps']
    df_pred['geo_pop_density_ntile'] = df_pred['zip_pop_dens']
    
    # CALC_VEH_AGE should already be calculated by apply_all_defaults
    if 'CALC_VEH_AGE' not in df_pred.columns:
        df_pred['CALC_VEH_AGE'] = model_year - df_pred['dml_year_imps']
    
    # Create STATE indicator columns
    states = df_pred['st_raw'].unique()
    for state in states:
        if pd.notna(state):
            df_pred[f'STATE_{state}'] = (df_pred['st_raw'] == state)
    
    # Create MAKE indicator columns
    makes = df_pred['dml_make_raw'].unique()
    for make in makes:
        if pd.notna(make):
            df_pred[f'MAKE_{make}'] = (df_pred['dml_make_raw'] == make)
    
    # Run prediction
    predictions = predict_glm(df_pred, glm_model)
    
    return predictions

In [ ]:
# Process ALL BSST segments
import time

print(f"{'='*80}")
print(f"PROCESSING ALL BSST SEGMENTS")
print(f"{'='*80}")
print(f"Total records: {len(df):,}")
print(f"BSST segments to process: {len(unique_bsst)}")
print()

# Initialize Dep_factor column with NaN
df['Dep_factor'] = pd.NA

start_time = time.time()
total_processed = 0
segments_processed = 0

for bsst in unique_bsst:
    # Load the GLM model for this BSST
    model_path = os.path.join(bsst_glm_path, f"{bsst}.json")
    
    if not os.path.exists(model_path):
        print(f"  ⚠ Model not found: {bsst} - skipping")
        continue
    
    # Get records for this BSST
    mask = df['BSST_formatted'] == bsst
    count = mask.sum()
    
    if count == 0:
        continue
    
    # Load model (suppress output for cleaner logs)
    with open(model_path, 'r') as f:
        glm_model = json.load(f)
    
    # Get segment data and run predictions
    df_segment = df[mask]
    predictions = prepare_and_predict(df_segment, glm_model, config['model_year'])
    
    # Assign predictions back to main dataframe
    df.loc[mask, 'Dep_factor'] = predictions.values
    
    total_processed += count
    segments_processed += 1
    
    # Progress update every 5 segments
    if segments_processed % 5 == 0 or segments_processed == len(unique_bsst):
        elapsed = time.time() - start_time
        print(f"  Processed {segments_processed}/{len(unique_bsst)} segments ({total_processed:,} records) - {elapsed:.1f}s")

# Calculate veh_value_dep
df['veh_value_dep'] = df['vc_msrp_impa'] * df['Dep_factor']

elapsed = time.time() - start_time
print(f"\n{'='*80}")
print(f"PROCESSING COMPLETE")
print(f"{'='*80}")
print(f"Total segments processed: {segments_processed}")
print(f"Total records with Dep_factor: {df['Dep_factor'].notna().sum():,}")
print(f"Records without Dep_factor (missing BSST): {df['Dep_factor'].isna().sum():,}")
print(f"Total time: {elapsed:.1f} seconds")

In [ ]:
# Summary statistics
print("\nDep_factor Statistics:")
print(df['Dep_factor'].describe())

print("\nveh_value_dep Statistics:")
print(df['veh_value_dep'].describe())

## Export Results to Parquet
Export selected columns to parquet file

In [ ]:
# Validate required columns exist before export
required_columns = ['ODOMETER_IMP_FLAG', 'CALC_VEH_AGE', 'Dep_factor', 'veh_value_dep']
missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    print(f"ERROR: Missing columns: {missing_cols}")
    print("Make sure you ran these cells in order:")
    print("  1. 'Apply all default values' cell (creates ODOMETER_IMP_FLAG, CALC_VEH_AGE)")
    print("  2. 'Process ALL BSST segments' cell (creates Dep_factor, veh_value_dep)")
    raise KeyError(f"Missing required columns: {missing_cols}")

print("✓ All required columns present in dataframe")
print(f"  Current df columns: {len(df.columns)}")

# Define output columns in order
output_columns = [
    'vin_date',                        # First column
    'vin',
    'BODY_STYLE_SEGMENT_BODY_TYPE',    # BSST
    'cef_est_curr_mi_grp_imps',        # ODOMETER (with defaults applied)
    'ODOMETER_IMP_FLAG',               # Flag for imputed odometer values
    'CALC_VEH_AGE',
    'st_raw',                          # STATE (with defaults applied)
    'vc_msrp_impa',
    'Dep_factor',
    'veh_value_dep'
]

# Create output dataframe with selected columns
df_output = df[output_columns].copy()

# Rename columns for clarity in output
df_output = df_output.rename(columns={
    'cef_est_curr_mi_grp_imps': 'ODOMETER',
    'BODY_STYLE_SEGMENT_BODY_TYPE': 'BSST',
    'st_raw': 'STATE'
})

print("Output columns:")
print(df_output.columns.tolist())
print(f"\nOutput shape: {df_output.shape}")
print(f"\nData types:")
print(df_output.dtypes)

In [ ]:
# Preview output
df_output.head(10)

In [ ]:
# Export to parquet
output_path = '/Users/Mach/dev/aps/data/2026_Dmodel_data/car_with_dep_factor.parquet'

print(f"Exporting to: {output_path}")
df_output.to_parquet(output_path, index=False)

# Verify export
file_size = os.path.getsize(output_path) / (1024 * 1024)  # Size in MB
print(f"\n✓ Export complete!")
print(f"  File size: {file_size:.2f} MB")
print(f"  Records: {len(df_output):,}")
print(f"  Columns: {len(df_output.columns)}")

In [ ]:
# Verify by reading back
df_verify = pd.read_parquet(output_path)
print(f"Verification - read back {len(df_verify):,} records")
print(f"\nODOMETER_IMP_FLAG distribution:")
print(df_verify['ODOMETER_IMP_FLAG'].value_counts())
print(f"\nRecords with imputed odometer: {(df_verify['ODOMETER_IMP_FLAG'] == 1).sum():,}")